# RadIntel AI — Pneumonia Detection: EDA + Baseline Training

Run this in **Google Colab** with a GPU runtime (`Runtime -> Change runtime type -> GPU`).

This notebook:
1. Downloads the RSNA Pneumonia Detection Challenge dataset from Kaggle
2. Does a quick EDA (class balance, sample images)
3. Trains the EfficientNet-B0 baseline using `train_efficientnet.py`
4. Runs Grad-CAM on a few test images to sanity-check the model


## 1. Setup

In [ ]:
!pip install -q timm pydicom albumentations grad-cam kaggle


### Kaggle API credentials
1. Go to https://www.kaggle.com/settings -> API -> "Create New Token" (downloads `kaggle.json`)
2. Upload it below when prompted


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload kaggle.json here

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json


In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p ./data
!unzip -q ./data/rsna-pneumonia-detection-challenge.zip -d ./data
!ls ./data


## 2. Quick EDA

In [ ]:
import pandas as pd

labels_df = pd.read_csv('./data/stage_2_train_labels.csv')
collapsed = labels_df.groupby('patientId', as_index=False)['Target'].max()

print(f"Total unique patients: {len(collapsed)}")
print(f"Pneumonia positive rate: {collapsed['Target'].mean():.3f}")
collapsed['Target'].value_counts().plot(kind='bar', title='Class balance (0=normal, 1=pneumonia)')


In [ ]:
import pydicom
import matplotlib.pyplot as plt

sample_ids = collapsed.sample(4, random_state=42)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (_, row) in zip(axes, sample_ids.iterrows()):
    dcm = pydicom.dcmread(f"./data/stage_2_train_images/{row['patientId']}.dcm")
    ax.imshow(dcm.pixel_array, cmap='gray')
    ax.set_title(f"Target: {row['Target']}")
    ax.axis('off')
plt.tight_layout()
plt.show()


## 3. Train the baseline model

Clone/upload the `ml/training/` folder's files (`dataset.py`, `augmentations.py`,
`train_efficientnet.py`, `configs/`) into this Colab session first, or mount
Google Drive if you're keeping the project there.


In [ ]:
# Adjust this path to wherever you placed the ml/training folder in Colab
%cd ml/training


In [ ]:
# Point the config at Colab's data location before training
import yaml

with open('configs/pneumonia_effnet_b0.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['images_dir'] = '/content/data/stage_2_train_images'
cfg['data']['labels_csv'] = '/content/data/stage_2_train_labels.csv'

with open('configs/pneumonia_effnet_b0.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)


In [ ]:
!python train_efficientnet.py --config configs/pneumonia_effnet_b0.yaml


## 4. Sanity-check with Grad-CAM

Run this on a few test images. The heatmap should highlight lung regions -
if it's consistently lighting up corners, borders, or text overlays instead,
that's a sign of a data/preprocessing issue worth investigating before
trusting the model's predictions.


In [ ]:
%cd ../inference
!python gradcam.py --checkpoint ../saved_models/efficientnet_b0_pneumonia.pt \
    --image /content/data/stage_2_train_images/<pick_a_patient_id>.dcm \
    --output gradcam_check.png

from PIL import Image
Image.open('gradcam_check.png')


## 5. Next step

Download `saved_models/efficientnet_b0_pneumonia.pt` from Colab, place it in
your backend at the path set by `MODEL_PATH` in `.env`, then set
`USE_MOCK_MODEL=false` and restart the backend. The `/predict` endpoint will
now use this real model instead of the mock random output.
